# 🚦 Traffic Sign Recognition — PYNQ-Z2 Demo Notebook

**Board:** PYNQ-Z2 (XC7Z020)  
**Dataset:** GTSRB (43 classes)  
**Model:** INT8 Quantized TFLite  
**Interface:** PYNQ Python API + AXI VDMA

---
### Requirements (run on the PYNQ-Z2 board)
Make sure these files are in `/home/xilinx/`:
- `design_1.bit` — bitstream
- `design_1.hwh` — hardware handoff
- `gtsrb_quantized.tflite` — quantized model

## Step 1 — Load the PYNQ Overlay

In [ ]:
from pynq import Overlay

# Load the hardware design (requires design_1.bit + design_1.hwh)
overlay = Overlay('design_1.bit')
print('✅ Overlay loaded successfully!')
print('Available IP blocks:', list(overlay.ip_dict.keys()))

## Step 2 — Access the VDMA (Camera Interface)

In [ ]:
# Access the AXI VDMA block from the overlay
# This matches the IP block name in the Vivado block design
vdma = overlay.axi_vdma_0

print('✅ VDMA block accessed')
print('VDMA read channel (S2MM):', vdma.readchannel)

## Step 3 — Load the TFLite Model

In [ ]:
import tflite_runtime.interpreter as tflite
import numpy as np

MODEL_PATH = 'gtsrb_quantized.tflite'
interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('✅ Model loaded:', MODEL_PATH)
print('Input  shape:', input_details[0]['shape'])
print('Input  dtype:', input_details[0]['dtype'])
print('Output shape:', output_details[0]['shape'])

## Step 4 — GTSRB Class Labels

In [ ]:
gtsrb_labels = {
    0: 'Speed limit (20km/h)',   1: 'Speed limit (30km/h)',   2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)',   4: 'Speed limit (70km/h)',   5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)',                         7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)',  9: 'No passing',
    10: 'No passing for vehicles over 3.5 metric tons',
    11: 'Right-of-way at the next intersection',              12: 'Priority road',
    13: 'Yield',                14: 'Stop',                   15: 'No vehicles',
    16: 'Vehicles over 3.5 metric tons prohibited',           17: 'No entry',
    18: 'General caution',      19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right',                       21: 'Double curve',
    22: 'Bumpy road',           23: 'Slippery road',          24: 'Road narrows on the right',
    25: 'Road work',            26: 'Traffic signals',         27: 'Pedestrians',
    28: 'Children crossing',    29: 'Bicycles crossing',       30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End of all speed and passing limits',                33: 'Turn right ahead',
    34: 'Turn left ahead',      35: 'Ahead only',             36: 'Go straight or right',
    37: 'Go straight or left',  38: 'Keep right',             39: 'Keep left',
    40: 'Roundabout mandatory', 41: 'End of no passing',
    42: 'End of no passing by vehicles over 3.5 metric tons'
}

print(f'✅ {len(gtsrb_labels)} GTSRB class labels loaded')

## Step 5 — Inference Helper Function

In [ ]:
import cv2
import time

def run_single_inference(frame):
    """Run one frame through the TFLite model and return label + confidence + latency."""
    # Pre-process
    img_rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (100, 100))

    if input_details[0]['dtype'] == np.float32:
        input_data = np.expand_dims(img_resized, axis=0).astype(np.float32) / 255.0
    else:
        input_data = np.expand_dims(img_resized, axis=0).astype(np.uint8)

    # Inference
    t0 = time.time()
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    latency_ms = (time.time() - t0) * 1000

    # Results
    output_data = interpreter.get_tensor(output_details[0]['index'])
    prediction  = int(np.argmax(output_data[0]))

    if output_details[0]['dtype'] == np.float32:
        confidence = float(output_data[0][prediction]) * 100
    else:
        confidence = float(output_data[0][prediction]) / 255.0 * 100

    label = gtsrb_labels.get(prediction, f'Class {prediction}')
    return label, confidence, latency_ms

print('✅ Inference function ready')

## Step 6 — Single Frame Capture & Inference

In [ ]:
from IPython.display import display
import PIL.Image
import io

# Capture one frame from the camera via VDMA
frame = vdma.readchannel.readframe()

# Run inference
label, confidence, latency_ms = run_single_inference(frame)

# Display the captured frame in the notebook
img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
pil_img = PIL.Image.fromarray(img_rgb)
display(pil_img)

print(f'\n🔍 Detected : {label}')
print(f'   Confidence: {confidence:.2f}%')
print(f'   Latency   : {latency_ms:.2f} ms')

## Step 7 — Continuous Inference Loop (10 frames)

In [ ]:
NUM_FRAMES = 10
results = []

print(f'Running inference on {NUM_FRAMES} frames...\n')
print(f'{"Frame":<6} {"Label":<45} {"Confidence":>12} {"Latency":>10}')
print('-' * 78)

for i in range(NUM_FRAMES):
    frame = vdma.readchannel.readframe()
    label, conf, lat = run_single_inference(frame)
    results.append({'frame': i+1, 'label': label, 'confidence': conf, 'latency_ms': lat})
    print(f'{i+1:<6} {label:<45} {conf:>10.2f}% {lat:>8.2f}ms')

avg_lat  = np.mean([r['latency_ms'] for r in results])
avg_conf = np.mean([r['confidence'] for r in results])
print('-' * 78)
print(f'{"AVERAGE":<52} {avg_conf:>10.2f}% {avg_lat:>8.2f}ms')
print(f'\nEstimated FPS: {1000/avg_lat:.1f}')

## Step 8 — Results Summary Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df['confidence'] = df['confidence'].map('{:.2f}%'.format)
df['latency_ms'] = df['latency_ms'].map('{:.2f} ms'.format)
df.columns = ['Frame', 'Detected Sign', 'Confidence', 'Latency']

display(df)

print('\n📊 Project Summary')
print(f'   Board         : PYNQ-Z2 (XC7Z020)')
print(f'   Model         : gtsrb_quantized.tflite (INT8)')
print(f'   Classes       : 43 GTSRB traffic signs')
print(f'   Avg Latency   : {avg_lat:.2f} ms')
print(f'   Avg Confidence: {avg_conf:.2f}%')
print(f'   Est. FPS      : {1000/avg_lat:.1f}')

## Step 9 — Cleanup

In [ ]:
# Free PYNQ resources
overlay.free()
print('✅ Overlay freed. Demo complete!')